# Heart Disease Classification (q1_supervised)
Build and evaluate classification models to predict `heart_disease`.

## 1. Data Loading and Inspection

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

df = pd.read_csv('q1_heart_disease.csv')
print("Shape:", df.shape)
print("\nData Types:\n", df.dtypes)
print("\nMissing Values:\n", df.isnull().sum())
df.head()


## 2. Exploratory Data Analysis

In [ ]:

plt.figure(figsize=(6,4))
sns.countplot(x='heart_disease', data=df)
plt.title('Target Class Distribution')
plt.show()


**Interpretation:** This chart shows whether the classes are balanced. A near-even split is ideal; noticeable imbalance may require stratified splitting (used later).

In [ ]:

plt.figure(figsize=(10,7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


**Interpretation:** Positive values indicate variables increasing together; negative values move oppositely. Features with stronger correlation to `heart_disease` may be informative predictors.

In [ ]:

plt.figure(figsize=(7,4))
sns.boxplot(x='heart_disease', y='max_hr', data=df)
plt.title('Max Heart Rate by Target Class')
plt.show()


**Interpretation:** Differences in median and spread of `max_hr` between classes suggest it may help discriminate patients with and without heart disease.

## 3. Data Preprocessing
Missing numeric values are imputed using the **median**, which is robust to outliers and avoids dropping rows unnecessarily.

In [ ]:

X = df.drop('heart_disease', axis=1)
y = df['heart_disease']

categorical_cols = X.select_dtypes(include='object').columns.tolist()
numeric_cols = X.select_dtypes(exclude='object').columns.tolist()

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_cols),
    ('cat', categorical_transformer, categorical_cols)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(X_train.shape, X_test.shape)


## 4. Model Training

In [ ]:

models = {
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[('prep', preprocessor), ('model', model)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    results[name] = {'pipeline': pipe, 'pred': y_pred}


## 5. Model Evaluation

In [ ]:

scores = []
for name, obj in results.items():
    y_pred = obj['pred']
    print(f"===== {name} =====")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=3))
    scores.append({
        'Model': name,
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1': f1_score(y_test, y_pred)
    })

pd.DataFrame(scores).sort_values('F1', ascending=False)


**Interpretation:** Select the best model primarily using **F1-score** (balance of precision and recall), while also reviewing precision and recall individually depending on whether false positives or false negatives matter more.

## 6. Hyperparameter Tuning

In [ ]:

# Choose the best model after reviewing previous results.
best_name = pd.DataFrame(scores).sort_values('F1', ascending=False).iloc[0]['Model']
best_estimator = models[best_name]

pipe = Pipeline(steps=[('prep', preprocessor), ('model', best_estimator)])

param_grid = {}
if best_name == 'Random Forest':
    param_grid = {
        'model__n_estimators': [100, 200],
        'model__max_depth': [None, 5, 10]
    }
elif best_name == 'Gradient Boosting':
    param_grid = {
        'model__n_estimators': [100, 200],
        'model__learning_rate': [0.05, 0.1],
        'model__max_depth': [2, 3]
    }
else:
    param_grid = {
        'model__max_depth': [None, 3, 5, 10],
        'model__min_samples_split': [2, 5, 10]
    }

grid = GridSearchCV(pipe, param_grid, cv=5, scoring='f1', n_jobs=-1)
grid.fit(X_train, y_train)

print("Best Model:", best_name)
print("Best Params:", grid.best_params_)

tuned_pred = grid.predict(X_test)
print("\nTuned Test Performance")
print(confusion_matrix(y_test, tuned_pred))
print(classification_report(y_test, tuned_pred, digits=3))


**Comparison:** Compare tuned model metrics to the untuned baseline. If F1-score improves, tuning was beneficial; otherwise the baseline may already generalize well.